
# Phase 3C.01 — Signal Construction

## Objectif de la séance

Construire un dataset de recherche permettant d'étudier le pouvoir prédictif du Time-Series Momentum sur notre univers de 16 ETF.

Cette séance couvre la préparation des données, le calcul du momentum, la construction des signaux directionnels et leur alignement avec les rendements futurs.

**Périmètre :** Train (2010–2017) et Validation (2018–2021). La période Test (2022–2025) reste exclue.


## 1. Environnement et imports

Importation des bibliothèques et des fonctions réutilisables développées dans les modules Python du projet.

In [41]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

## 2. Chargement des données

Récupération des prix de clôture ajustés des ETF via `download_prices()`, définie dans `src/data.py`.

L'historique commence en 2008 afin de disposer des données nécessaires à la construction des premiers signaux en 2010. Les données s'arrêtent à décembre 2021 pour préserver notre période Test.

In [44]:
from src.data import download_prices

asset_universe = {
    "Equities" : ["SPY", "VGK", "EWJ", "EEM"],
    "Commodities" : ["USO", "DBA", "DBB", "DBC"],
    "Bonds" : ["IEF", "TLT", "LQD", "HYG"],
    "Forex" : ["FXE", "FXY", "FXB", "FXA"]
}

tickers = []

for assets in asset_universe.values():
    tickers.extend(assets)

print(tickers)

prices = download_prices(tickers, "2008-01-01", "2022-01-01")

print(prices.head())
print(prices.shape)
print(prices.columns)

['SPY', 'VGK', 'EWJ', 'EEM', 'USO', 'DBA', 'DBB', 'DBC', 'IEF', 'TLT', 'LQD', 'HYG', 'FXE', 'FXY', 'FXB', 'FXA']
Ticker            DBA        DBB        DBC        EEM        EWJ        FXA  \
Date                                                                           
2008-01-02  28.197397  18.034830  26.991880  33.408066  38.190758  64.639412   
2008-01-03  28.863747  18.908264  27.199961  33.708649  38.335087  64.434677   
2008-01-04  29.167398  18.455370  27.041821  32.696136  37.353611  63.754585   
2008-01-07  28.779396  18.544333  26.550756  32.935719  37.122684  63.761887   
2008-01-08  29.099916  19.207495  26.792126  32.680325  37.036091  64.456619   

Ticker             FXB         FXE        FXY        HYG        IEF  \
Date                                                                  
2008-01-02  172.158783  134.518661  91.190002  31.209982  55.187588   
2008-01-03  171.550674  134.884048  91.400002  31.150389  55.300991   
2008-01-04  171.515930  134.957169  92.110


## 3. Validation des données

Cette étape consiste à vérifier l'intégrité des données téléchargées avant leur utilisation dans la recherche.

Les contrôles portent sur la couverture temporelle, les dimensions du dataset, les valeurs manquantes, les dates dupliquées et les prix non positifs.


In [45]:
print(prices.shape) # Dimensions du DataFrame
print(prices.index.min()) # Première date disponible
print(prices.index.max()) # Dernière date disponible
print(len(prices.columns)) # Nombre d'ETF
print(prices.isna().sum()) # Nombre de valeurs manquantes par actifs
print(prices.index.duplicated().sum()) # Nombre de dates dupliquées
print((prices <= 0).sum()) # Nombre de prix inferieur ou égale a zéro

(3526, 16)
2008-01-02 00:00:00
2021-12-31 00:00:00
16
Ticker
DBA    0
DBB    0
DBC    0
EEM    0
EWJ    0
FXA    0
FXB    0
FXE    0
FXY    0
HYG    0
IEF    0
LQD    0
SPY    0
TLT    0
USO    0
VGK    0
dtype: int64
0
Ticker
DBA    0
DBB    0
DBC    0
EEM    0
EWJ    0
FXA    0
FXB    0
FXE    0
FXY    0
HYG    0
IEF    0
LQD    0
SPY    0
TLT    0
USO    0
VGK    0
dtype: int64



### Conclusion — Validation des données

L'univers de recherche contient 16 ETF répartis entre quatre classes d'actifs.

Les données couvrent la période de janvier 2008 à décembre 2021, incluant l'historique nécessaire au calcul des premiers signaux.

Les contrôles d'intégrité ne révèlent aucune valeur manquante, aucun doublon de date et aucun prix non positif.

La période Test (2022–2025) reste exclue de l'analyse.



## 4. Préparation des données mensuelles

Conversion des prix quotidiens en prix de fin de mois à l'aide de la fonction `to_monthly_prices()` définie dans `src/data.py`.

Cette fréquence correspond à notre protocole de recherche, qui prévoit des signaux de momentum mensuels.


In [53]:
import importlib
import src.data

importlib.reload(src.data) # On force la relecture de src.data car le programme ne trouvais pas to_monthly_prices dans src.data car ajouter après le premier import.

from src.data import to_monthly_prices

monthly_prices = to_monthly_prices(prices)
print(monthly_prices.head())
print(monthly_prices.shape)
print(monthly_prices.index.min())
print(monthly_prices.index.max())
print(monthly_prices.isna().sum())
print(monthly_prices.index.duplicated().sum())

Ticker            DBA        DBB        DBC        EEM        EWJ        FXA  \
Date                                                                           
2008-01-31  31.200172  19.611864  27.033499  30.937838  36.718540  65.955765   
2008-02-29  35.054863  21.755016  30.046467  31.554817  36.170078  68.886528   
2008-03-31  30.744696  20.388252  29.771803  30.370556  35.708218  67.821236   
2008-04-30  30.871216  20.388252  31.569601  33.145897  38.335087  70.361084   
2008-05-31  29.816872  19.385418  33.750252  34.194561  39.085629  71.783920   

Ticker             FXB         FXE         FXY        HYG        IEF  \
Date                                                                   
2008-01-31  173.548584  136.327179   93.910004  31.015533  56.636086   
2008-02-29  173.889603  139.494537   96.129997  30.435730  57.337944   
2008-03-31  174.371017  145.382812  100.239998  30.723680  58.104778   
2008-04-30  175.323532  144.189941   95.949997  31.767324  56.700085   
2008-05


### Conclusion — Préparation des données mensuelles

Les prix quotidiens ont été convertis en prix de fin de mois à l'aide de la fonction `to_monthly_prices()` définie dans `src/data.py`.

Le dataset contient **168 observations mensuelles et 16 ETF**, couvrant janvier 2008 à décembre 2021.

Aucune valeur manquante ni date dupliquée n'a été détectée après la conversion.

Les données sont prêtes pour la construction des signaux de Time-Series Momentum.



## 5. Calcul du momentum

Calcul des rendements passés sur différentes fenêtres temporelles à l'aide de la fonction `calculate_momentum()` définie dans `src/signals.py`.

L'horizon principal de notre recherche est de 12 mois. Les horizons de 3 et 6 mois serviront aux analyses de robustesse.


In [79]:
import importlib
import src.signals

importlib.reload(src.signals)

from src.signals import calculate_momentum

momentum_12m = calculate_momentum(monthly_prices, lookback=12)
print(momentum_12m.head(24))

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2008-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-02-29       NaN       NaN       NaN       NaN       NaN       NaN   
2008-03-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-04-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-05-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-06-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-07-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-08-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-09-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-10-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-11-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-12-31       NaN       NaN       N


### 5.1. Vérification de cohérence du momentum

L'objectif est de vérifier que la fonction `calculate_momentum()` calcule correctement le rendement cumulé sur 12 mois.

Nous comparons le momentum calculé automatiquement pour SPY au 31 janvier 2009 avec un calcul direct à partir des prix de janvier 2008 et janvier 2009.

Les deux résultats doivent être identiques, à la précision numérique près.


In [116]:
prices_start = monthly_prices.loc["2008-01-31", "SPY"]
prices_end = monthly_prices.loc["2009-01-31", "SPY"]

print(prices_start)
print(prices_end)

manual_momentum = (prices_end / prices_start) - 1 
print(manual_momentum)
print(momentum_12m["SPY"].loc["2009-01-31"])
sanity_check = manual_momentum - momentum_12m["SPY"].loc["2009-01-31"]
sanity_check

97.60250854492188
60.267921447753906
-0.3825166755830318
-0.3825166755830318


np.float64(0.0)


### Conclusion — Calcul du momentum

La fonction `calculate_momentum()` calcule le rendement cumulé sur une fenêtre historique paramétrable.

Le momentum à 12 mois de SPY au 31 janvier 2009 a été vérifié par un calcul indépendant.

- Momentum manuel : **−38,25 %**
- Momentum calculé par la fonction : **−38,25 %**
- Écart : **0**

Le contrôle de cohérence est validé. Les 12 premières observations restent manquantes en raison de la période d'initialisation (*warm-up*).

## 6. Construction des signaux directionnels

Transformation du momentum à 12 mois en signaux directionnels :
+1 pour un momentum positif, -1 pour un momentum négatif et 0 pour un momentum nul.

Les observations manquantes sont conservées.

In [126]:
import importlib
import src.signals

importlib.reload(src.signals)

from src.signals import generate_signals

signals_12m = generate_signals(momentum_12m)

print(signals_12m.head())
print(signals_12m.loc["2009-01-31"])

Ticker      DBA  DBB  DBC  EEM  EWJ  FXA  FXB  FXE  FXY  HYG  IEF  LQD  SPY  \
Date                                                                          
2008-01-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-02-29  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-03-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-04-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-05-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   

Ticker      TLT  USO  VGK  
Date                       
2008-01-31  NaN  NaN  NaN  
2008-02-29  NaN  NaN  NaN  
2008-03-31  NaN  NaN  NaN  
2008-04-30  NaN  NaN  NaN  
2008-05-31  NaN  NaN  NaN  
Ticker
DBA   -1.0
DBB   -1.0
DBC   -1.0
EEM   -1.0
EWJ   -1.0
FXA   -1.0
FXB   -1.0
FXE   -1.0
FXY    1.0
HYG   -1.0
IEF    1.0
LQD   -1.0
SPY   -1.0
TLT    1.0
USO   -1.0
VGK   -1.0
Name: 2009-01-31 00:00:00, dtype: float64



### Conclusion — Construction des signaux

La fonction `generate_signals()` transforme le momentum en prédictions directionnelles à l'aide de `np.sign()`.

Les observations manquantes de la période de warm-up sont conservées.

Les signaux sont construits uniquement à partir des informations historiques disponibles à chaque date.


## 7. Construction des rendements futurs

Calcul du rendement réalisé au cours du mois suivant, aligné sur la date de formulation du signal.

In [136]:
forward_returns_1m = (monthly_prices.pct_change()).shift(-1)
print(forward_returns_1m.head())
print(forward_returns_1m.tail())

prices_january = monthly_prices.loc["2008-01-31", "SPY"]
prices_february = monthly_prices.loc["2008-02-29", "SPY"]

manual_forward_returns_1m = (prices_february / prices_january) -1
print(manual_forward_returns_1m)
verification = (forward_returns_1m.loc["2008-01-31", "SPY"] - manual_forward_returns_1m)
print(verification)

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2008-01-31  0.123547  0.109278  0.111453  0.019943 -0.014937  0.044435   
2008-02-29 -0.122955 -0.062825 -0.009141 -0.037530 -0.012769 -0.015464   
2008-03-31  0.004115  0.000000  0.060386  0.091383  0.073565  0.037449   
2008-04-30 -0.034153 -0.049187  0.069074  0.031638  0.019578  0.020222   
2008-05-31  0.150778  0.043388  0.104316 -0.092695 -0.074608  0.008130   

Ticker           FXB       FXE       FXY       HYG       IEF       LQD  \
Date                                                                     
2008-01-31  0.001965  0.023234  0.023640 -0.018694  0.012392 -0.007062   
2008-02-29  0.002769  0.042212  0.042755  0.009461  0.013374 -0.005972   
2008-03-31  0.005463 -0.008205 -0.042797  0.033969 -0.024175  0.012447   
2008-04-30  0.000468 -0.000179 -0.013549 -0.005085 -0.017774 -0.022052   
2008-05-31  0.008562  0.014124 -0.007


### Conclusion — Construction des rendements futurs

Les rendements futurs à un mois ont été calculés à partir des prix mensuels avec `.pct_change().shift(-1)`.

Un contrôle de cohérence a été réalisé sur SPY au 31 janvier 2008 :

- Rendement futur calculé : **−2,5842 %**
- Rendement manuel : **−2,5842 %**
- Écart : **0**

Le calcul et l'alignement temporel sont validés.

La dernière observation reste manquante, car le prix du mois suivant n'est pas disponible dans notre périmètre de données.



## 8. Construction des targets directionnelles

### Objectif

Transformer les rendements futurs à un mois (`forward_returns_1m`) en variables cibles directionnelles (`targets_1m`).

La transformation est réalisée à l'aide de la fonction `generate_signals()` :

- **+1** : rendement futur positif (hausse).
- **-1** : rendement futur négatif (baisse).
- **0** : rendement futur nul.
- **NaN** : rendement futur indisponible.

Les targets représentent la direction réellement observée le mois suivant. Elles permettront d'évaluer les prédictions issues du momentum.


In [141]:
targets_1m = generate_signals(forward_returns_1m)
print(targets_1m)
print(targets_1m.loc["2008-01-31", "SPY"])
print(targets_1m.loc["2021-12-31", "SPY"])

Ticker      DBA  DBB  DBC  EEM  EWJ  FXA  FXB  FXE  FXY  HYG  IEF  LQD  SPY  \
Date                                                                          
2008-01-31  1.0  1.0  1.0  1.0 -1.0  1.0  1.0  1.0  1.0 -1.0  1.0 -1.0 -1.0   
2008-02-29 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0  1.0  1.0  1.0  1.0  1.0 -1.0 -1.0   
2008-03-31  1.0  0.0  1.0  1.0  1.0  1.0  1.0 -1.0 -1.0  1.0 -1.0  1.0  1.0   
2008-04-30 -1.0 -1.0  1.0  1.0  1.0  1.0  1.0 -1.0 -1.0 -1.0 -1.0 -1.0  1.0   
2008-05-31  1.0  1.0  1.0 -1.0 -1.0  1.0  1.0  1.0 -1.0 -1.0  1.0 -1.0 -1.0   
...         ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   
2021-08-31 -1.0 -1.0  1.0 -1.0  1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0   
2021-09-30  1.0  1.0  1.0  1.0 -1.0  1.0  1.0 -1.0 -1.0 -1.0 -1.0  1.0  1.0   
2021-10-31  1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0  1.0 -1.0  1.0 -1.0 -1.0   
2021-11-30  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0 -1.0  1.0 -1.0 -1.0  1.0   
2021-12-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  


### Conclusion — Construction des targets

Les rendements futurs ont été transformés en directions à l'aide de la fonction `generate_signals()`.

Les targets prennent les valeurs +1 (hausse), -1 (baisse), 0 (rendement nul) ou NaN (observation indisponible).

Le contrôle effectué sur SPY confirme la cohérence des résultats.



## 9. Vérification de l'alignement des données

### Objectif

Vérifier que les prédictions (`signals_12m`) et les observations réelles (`targets_1m`) possèdent les mêmes dimensions, dates et colonnes.

Cette vérification permet de s'assurer que chaque prédiction est associée au résultat futur du même ETF, à la même date de formulation du signal.

Les contrôles portent sur :

- Les dimensions des deux DataFrames.
- L'égalité des index temporels.
- L'égalité et l'ordre des colonnes.

Un alignement correct est indispensable pour éviter des erreurs méthodologiques lors de l'évaluation prédictive.


In [ ]:
print(signals_12m.shape)
print(targets_1m.shape)

print(signals_12m.index.equals(targets_1m.index))
print(signals_12m.columns.equals(targets_1m.columns))

(168, 16)
(168, 16)
True
True



### Conclusion — Alignement des données

Les DataFrames `signals_12m` et `targets_1m` possèdent les mêmes dimensions (168 × 16), les mêmes dates et les mêmes colonnes.

L'alignement temporel et structurel est validé.

Les données peuvent désormais être préparées pour l'analyse prédictive.



## 10. Préparation des échantillons Train & Validation

### Objectif

Séparer les données en périodes d'apprentissage et de validation conformément au protocole expérimental de la Phase 3B.

- **Train :** 2010–2017.
- **Validation :** 2018–2021.
- **Test :** 2022–2025, réservé à la Phase 3F.

### 10.1. Découpage initial du Train

Extraction des signaux et des targets sur la période du 31 janvier 2010 au 31 décembre 2017.

Vérification des dimensions et de l'alignement temporel.

**Précaution méthodologique :** les targets correspondent aux rendements du mois suivant. Les frontières des périodes devront être ajustées pour éviter d'attribuer au Train un rendement réalisé pendant la Validation.

Les observations manquantes et les signaux neutres sont conservés pour le moment.


In [ ]:
train_signals = signals_12m.loc["2009-12-31":"2017-11-30"]
train_targets = targets_1m.loc["2009-12-31":"2017-11-30"]

print(train_signals.shape)
print(train_targets.shape)
print(train_signals.index.equals(train_targets.index))

(96, 16)
(96, 16)
True



### 10.2. Découpage de la Validation

Extraction des signaux et des targets correspondant aux rendements réalisés entre janvier 2018 et décembre 2021.

Les dates des signaux sont décalées d'un mois par rapport aux périodes de rendement afin de respecter l'alignement prédiction–observation.

L'échantillon Validation doit contenir 48 observations mensuelles pour chacun des 16 ETF.


In [158]:
validation_signals = signals_12m.loc["2017-12-31":"2021-11-30"]
validation_targets = targets_1m.loc["2017-12-31":"2021-11-30"]

print(validation_signals.shape)
print(validation_targets.shape)
print(validation_signals.index.equals(validation_targets.index))

dates_verif = (len(train_signals.index.intersection(validation_signals.index)))
dates_verif

(48, 16)
(48, 16)
True


0